# Stage 14 - more training data for the fine-tuned reranker

Mines three new shards, trains the Stage 8 recipe from the base model on 4k and
10k groups, and scores both against the Stage 8 weights on the Stage 6 bench.
Details: `docs/stage14_data_scale.md`.

## Setup

In [ ]:
# project folder on Drive; every account must see the shared folder at this path
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'
ACCOUNT_LABEL = 'A'          # recorded in the run lock and the progress log
CLEAR_STALE_LOCK = False     # True only after confirming the runtime holding the lock is stopped
MODE = 'fresh'               # 'resume' to continue a run started earlier

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - the shared folder is not mounted at this path.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')


def run(cmd):
    """Stream the command's output live; a failure stops Run All."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

Shared-folder id, write access, GPU, run lock and progress so far.

In [ ]:
run('python -u scripts/35_preflight_stage14.py --gpu')

## Mine the shards

Each shard's groups are written when it finishes; finished shards are skipped.

In [ ]:
run('python -u scripts/32_build_stage14_data.py' + GUARD)

## Train the 4k and 10k arms

A checkpoint is saved every 500 steps and each save prints a line saying it is safe
to stop. Finished arms are skipped.

In [ ]:
run(f'python -u scripts/33_train_data_scale.py --mode {MODE}' + GUARD)

## Dev bench

Provenance only.

In [ ]:
run('python -u scripts/34_eval_data_scale.py --dev' + GUARD)

## Stage 6 bench

Fixed 15/0, four rerankers over the same pool, checkpointed per config.

In [ ]:
run('python -u scripts/34_eval_data_scale.py' + GUARD)

## Results

In [ ]:
import json, pathlib
from IPython.display import Markdown, display

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
summary = latest / 'stage14_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    print(json.dumps(json.loads((latest / 'stage14_verdict.json').read_text(encoding='utf-8')),
                     indent=2))
else:
    print('no Stage 14 summary - the evaluation did not finish.')

## Verdicts

`INVALID`, `INCONCLUSIVE-SIZE`, `MORE-DATA-BETTER`, `MORE-DATA-WORSE` and `TIE`
are defined in `docs/stage14_data_scale.md`. Archiving is manual:
`results/latest/` also holds other stages' files.